# QLoRA fine-tuning: Qwen2.5-1.5B-Instruct on SciTLDR with citations

Runs top to bottom on a single **T4 (16 GB)** in Kaggle or Colab. It teaches the model to summarize tagged paper chunks and to cite the chunk it used as `[Doc N, Sec S]`.

Output: `qlora_adapter.zip`. Unzip it into `research_assistant/data/models/qlora_adapter/` and set `LLM_PROVIDER=local`.

Versions are pinned to the ones this notebook was written against (trl 1.18 API: `SFTConfig(max_length=...)`, float `warmup_steps` as a ratio).

In [ ]:
%pip install -q "transformers==5.23.0" "trl==1.18.2" "peft==0.22.0" "datasets==5.2.0" accelerate bitsandbytes rouge-score

In [ ]:
import random, re, shutil

import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from rouge_score import rouge_scorer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), "Enable a GPU runtime (T4)."
print(torch.cuda.get_device_name(0))

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SYSTEM = "You are a research assistant. Cite sources as [Doc N, Sec S]."
N_TRAIN, N_EVAL, N_ROUGE = 2000, 200, 50
OUT_DIR = "qlora_adapter"
SEED = 42
random.seed(SEED); torch.manual_seed(SEED)

## Data: SciTLDR (Abstract) → tagged chunks + cited TLDR

`allenai/scitldr` is a script-based dataset, which `datasets` 5.x can't run, so we load Hugging Face's auto-converted parquet files.
The train split has only 1,992 rows, so training draws from train + test, and eval uses validation.
Each abstract (a list of sentences) is split into 2 sections, or 3 if it has 6+ sentences. The citation points at the section holding the sentence with the highest ROUGE overlap with the TLDR (SciTLDR's `rouge_scores`, or word overlap when those are missing).

In [ ]:
PARQUET = "hf://datasets/allenai/scitldr@refs/convert/parquet/Abstract/{split}/0000.parquet"
raw = load_dataset("parquet", data_files={s: PARQUET.format(split=s) for s in ["train", "test", "validation"]})
print(raw)

def split_sections(sentences):
    n = 3 if len(sentences) >= 6 else 2
    n = min(n, len(sentences))
    size = -(-len(sentences) // n)  # ceil
    return [sentences[i:i + size] for i in range(0, len(sentences), size)]

CITE_RE = re.compile(r"\s*\[Doc \d+, Sec \d+\]")

def to_example(row):
    sections = split_sections(row["source"])
    scores = row["rouge_scores"]
    if len(scores) != len(row["source"]):  # some rows lack per-sentence scores: use word overlap
        tldr = set(row["target"][0].lower().split())
        scores = [len(tldr & set(s.lower().split())) for s in row["source"]]
    best = max(range(len(scores)), key=scores.__getitem__)
    sec_of = [s for s, sents in enumerate(sections, 1) for _ in sents]
    user = "Summarize this paper in one sentence (TLDR) and cite the section you used.\n\n" + "\n\n".join(
        f"[Doc 1, Sec {s}] " + " ".join(sents) for s, sents in enumerate(sections, 1)
    )
    answer = f"{row['target'][0].strip()} [Doc 1, Sec {sec_of[best]}]"
    return {
        "prompt": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}],
        "completion": [{"role": "assistant", "content": answer}],
    }

def build(splits, n):
    rows = [r for s in splits for r in raw[s] if r["source"] and r["target"]]
    random.Random(SEED).shuffle(rows)
    return Dataset.from_list([to_example(r) for r in rows[:n]])

# The train split has only 1,992 papers, so train + test (2,610) supply the 2,000 training examples.
train_ds, eval_ds = build(["train", "test"], N_TRAIN), build(["validation"], N_EVAL)
print(len(train_ds), len(eval_ds))
print(eval_ds[0]["prompt"][1]["content"][:600], "\n→", eval_ds[0]["completion"][0]["content"])

## Model: 4-bit NF4 + double quant, fp16 compute (the T4 has no bf16)

In [ ]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, dtype=torch.float16, device_map={"": 0}
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
print(f"{torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")

## Baseline ROUGE-L (before training)

We use greedy decoding on the first 50 eval examples. ROUGE-L is computed on the TLDR text with citation tags stripped, and citation accuracy is reported separately.

In [ ]:
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

@torch.no_grad()
def generate(m, examples, batch_size=8, max_new_tokens=96):
    tokenizer.padding_side = "left"
    m.eval()
    outs = []
    for i in range(0, len(examples), batch_size):
        prompts = [tokenizer.apply_chat_template(e["prompt"], tokenize=False, add_generation_prompt=True)
                   for e in examples[i:i + batch_size]]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to(m.device)
        gen = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
        outs += tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [o.strip() for o in outs]

def score(preds, examples):
    refs = [e["completion"][0]["content"] for e in examples]
    rouge = sum(scorer.score(CITE_RE.sub("", r), CITE_RE.sub("", p))["rougeL"].fmeasure
                for p, r in zip(preds, refs)) / len(refs)
    cite = sum(CITE_RE.search(p) is not None and CITE_RE.search(p).group().strip() == CITE_RE.search(r).group().strip()
               for p, r in zip(preds, refs)) / len(refs)
    return rouge, cite

rouge_examples = [eval_ds[i] for i in range(N_ROUGE)]
base_preds = generate(model, rouge_examples)
base_rouge, base_cite = score(base_preds, rouge_examples)
print(f"Base  ROUGE-L {base_rouge:.4f} | citation acc {base_cite:.2%}")

## Train with LoRA (r=16, α=32) on all attention and MLP projections

The data is in prompt/completion form, so the loss is computed on the assistant completion only.

In [ ]:
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
args = SFTConfig(
    output_dir="sft_out",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=0.03,  # float < 1 = ratio of total steps
    max_length=1024,
    gradient_checkpointing=True,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=SEED,
)
tokenizer.padding_side = "right"
trainer = SFTTrainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
    processing_class=tokenizer, peft_config=peft_config,
)
trainer.model.print_trainable_parameters()
trainer.train()

## Samples and ROUGE-L: base vs. fine-tuned

In [ ]:
ft_model = trainer.model
ft_preds = generate(ft_model, rouge_examples)
ft_rouge, ft_cite = score(ft_preds, rouge_examples)

for i in range(3):
    print(f"--- Example {i + 1} ---")
    print("REFERENCE :", rouge_examples[i]["completion"][0]["content"])
    print("BASE      :", base_preds[i])
    print("FINE-TUNED:", ft_preds[i], "\n")

print(f"{'model':<12}{'ROUGE-L':>10}{'cite acc':>10}")
print(f"{'base':<12}{base_rouge:>10.4f}{base_cite:>10.2%}")
print(f"{'fine-tuned':<12}{ft_rouge:>10.4f}{ft_cite:>10.2%}")

## Save the adapter as a zip

The zip holds the adapter and tokenizer files at its root, so you can unzip it straight into `data/models/qlora_adapter/`.

In [ ]:
trainer.model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
zip_path = shutil.make_archive(OUT_DIR, "zip", root_dir=OUT_DIR)
print("Saved", zip_path)
try:  # Colab: trigger a browser download. Kaggle: grab it from the Output panel (/kaggle/working).
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass